# Medical Chatbot Fine-Tuning with Mistral 7B + LoRA

This notebook fine-tunes Mistral-7B-Instruct on a real medical Q&A dataset from Hugging Face using LoRA.

**Dataset Options:**
- **medalpaca/medical_meadow_medqa** - Medical Q&A pairs
- **GBaker/MedQA** - USMLE-style medical questions
- **pubmed_qa** - PubMed-based Q&A

**Model:** Mistral-7B-Instruct (fully open source, no license approval needed)

## 1. Setup Environment

In [2]:
# Install required packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets accelerate peft bitsandbytes trl
!pip install -q langchain langchain-community langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    pipeline,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset
from trl import SFTTrainer
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.64 GB


## 2. Load Pre-trained Mistral 7B Model

In [4]:
# Model configuration - Mistral 7B Instruct (fully open source!)
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

# Quantization config to reduce memory usage (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Mistral tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Mistral-7B with 4-bit quantization...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f"\n✓ Model loaded! Parameters: {base_model.num_parameters() / 1e9:.2f}B")
print("No license approval needed - ready to go!")

Loading Mistral tokenizer...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


✓ Model loaded! Parameters: 7.25B
No license approval needed - ready to go!


## 3. Test Model BEFORE Fine-tuning

In [5]:
def generate_response(model, tokenizer, question, max_new_tokens=200):
    """Generate response from Mistral model"""
    # Mistral instruction format
    prompt = f"""<s>[INST] {question} [/INST]"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the response part (after [/INST])
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()
    return response

In [6]:
# The question you specified
test_question = "If suddenly when I walk my stomach started to hurt me, what do you think it is and what is your suggestion?"

print("="*80)
print("RESPONSE BEFORE FINE-TUNING (Base Mistral Model)")
print("="*80)
response_before = generate_response(base_model, tokenizer, test_question)
print(f"\nQuestion: {test_question}\n")
print(f"Response:\n{response_before}")
print("="*80)

RESPONSE BEFORE FINE-TUNING (Base Mistral Model)

Question: If suddenly when I walk my stomach started to hurt me, what do you think it is and what is your suggestion?

Response:
If suddenly when I walk my stomach started to hurt me, what do you think it is and what is your suggestion?  I'm not a doctor, but I can suggest some common causes of stomach pain and advise you to seek medical attention if the pain persists. Here are some possibilities:

1. Indigestion or gastritis: This can be caused by eating spicy or fatty foods, stress, or certain medications. Symptoms may include bloating, nausea, and a burning sensation in the stomach.

2. Food poisoning: Consuming contaminated food or water can lead to food poisoning, which can cause stomach pain, diarrhea, vomiting, and fever.

3. Ulcers: Stomach ulcers are sores that develop in the lining of the stomach or small intestine. Symptoms may include burning or gnawing pain in the abdomen, especially between meals or at night, and weight lo

## 4. Load Medical Q&A Dataset from Hugging Face

In [12]:



DATASET_NAME = "FreedomIntelligence/medical-o1-reasoning-SFT"
TEXT_FIELD = "Question"  # Field containing the question
COT_FIELD="Complex_CoT"
OUTPUT_FIELD = "Response"  # Field containing the answer


print(f"Loading dataset: {DATASET_NAME}")
print("This may take a few minutes...\n")

Loading dataset: FreedomIntelligence/medical-o1-reasoning-SFT
This may take a few minutes...



In [13]:
# Load dataset from Hugging Face
dataset = load_dataset(DATASET_NAME,name="en" ,split="train")

print(f"✓ Dataset loaded!")
print(f"Total samples: {len(dataset):,}")
print(f"\nDataset columns: {dataset.column_names}")
print(f"\nSample entry:")
print("-" * 80)
sample = dataset[0]
for key, value in sample.items():
    print(f"{key}: {str(value)[:200]}..." if len(str(value)) > 200 else f"{key}: {value}")

✓ Dataset loaded!
Total samples: 19,704

Dataset columns: ['Question', 'Complex_CoT', 'Response']

Sample entry:
--------------------------------------------------------------------------------
Question: Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to ...
Complex_CoT: Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?

But wait, there's more. The right lower le...
Response: The specific cardiac abnormality most likely to be found in this scenario is a patent foramen ovale (PFO). This condition could allow a blood clot from the venous system, such as one from a deep vein ...


In [14]:

def prepare_medical_data(sample):
    """Extract question and direct response for clean training"""
    question = sample.get('Question', '')
    response = sample.get('Response', '')

    return {
        "question": question,
        "answer": response
    }


dataset = dataset.map(prepare_medical_data, remove_columns=dataset.column_names)


dataset = dataset.filter(lambda x: len(x['question']) > 10 and len(x['answer']) > 10)


MAX_SAMPLES = 500
if len(dataset) > MAX_SAMPLES:
    dataset = dataset.select(range(MAX_SAMPLES))

split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"\n✓ Dataset prepared successfully (Direct Q&A)!")
print(f"Training samples: {len(split_dataset['train'])}")
print(f"Validation samples: {len(split_dataset['test'])}")


def format_instruction(sample):
    """Format data in Mistral instruction style"""
    return f"<s>[INST] {sample['question']} [/INST] {sample['answer']} </s>"


sample_formatted = format_instruction(split_dataset['train'][0])
print(f"\nFormatted Sample Preview:\n" + "-"*80)
print(sample_formatted)


✓ Dataset prepared successfully (Direct Q&A)!
Training samples: 400
Validation samples: 100

Formatted Sample Preview:
--------------------------------------------------------------------------------
<s>[INST] In a patient with wrist trauma resulting in a sprained wrist and tenderness in the anatomical snuffbox, which specific ligament is most commonly involved when there is no evidence of a fracture? [/INST] In a patient with wrist trauma resulting in a sprained wrist and tenderness in the anatomical snuffbox, where there is no evidence of a fracture, the scapholunate ligament is most commonly involved. This ligament connects the scaphoid and lunate bones and is frequently implicated when there is tenderness in the snuffbox without a fracture. </s>


In [15]:
# Visualize a few samples
print("\n" + "="*80)
print("DATASET SAMPLES")
print("="*80)

for i in range(3):
    sample = split_dataset['train'][i]
    print(f"\n--- Sample {i+1} ---")
    print(f"Q: {sample['question'][:200]}")
    print(f"A: {sample['answer'][:200]}...")
print("="*80)


DATASET SAMPLES

--- Sample 1 ---
Q: In a patient with wrist trauma resulting in a sprained wrist and tenderness in the anatomical snuffbox, which specific ligament is most commonly involved when there is no evidence of a fracture?
A: In a patient with wrist trauma resulting in a sprained wrist and tenderness in the anatomical snuffbox, where there is no evidence of a fracture, the scapholunate ligament is most commonly involved. T...

--- Sample 2 ---
Q: What is the most likely diagnosis for a patient who presents with respiratory symptoms such as cough and hemoptysis, along with glomerulonephritis, and has raised c-ANCA levels in the serum?
A: The most likely diagnosis for a patient who presents with respiratory symptoms such as cough and hemoptysis, along with glomerulonephritis, and has raised c-ANCA levels in the serum is Granulomatosis ...

--- Sample 3 ---
Q: Which permanent tooth is most challenging to differentiate between mesial and distal aspects based on its morphology?
A:

## 5. Configure LoRA (Parameter Efficient Fine-Tuning)

In [16]:
# LoRA Configuration optimized for Mistral
lora_config = LoraConfig(
    r=16,                          # LoRA rank
    lora_alpha=32,                 # LoRA scaling parameter
    lora_dropout=0.05,             # Dropout probability
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[               # Mistral attention modules
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

# Apply LoRA to model
print("Applying LoRA adapters to Mistral...")
model = get_peft_model(base_model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\n{'='*60}")
print(f"LoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"\nTrainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.4f}%)")
print(f"Total parameters: {total_params:,}")
print(f"{'='*60}")

Applying LoRA adapters to Mistral...

LoRA Configuration:
  Rank (r): 16
  Alpha: 32
  Dropout: 0.05

Trainable parameters: 41,943,040 (1.1037%)
Total parameters: 3,800,305,664


## 6. Fine-Tuning Training

In [17]:


# Training arguments - optimized for free tier Colab
training_args = TrainingArguments(
    output_dir="./medical-chatbot-mistral-lora",
    num_train_epochs=2,                    # 2 epochs
    per_device_train_batch_size=1,         # Small batch for 7B model
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,         # Simulate larger batch
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    warmup_steps=10,
    max_grad_norm=1.0,
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

Training configuration:
  Epochs: 2
  Batch size: 1
  Gradient accumulation: 8
  Effective batch size: 8
  Learning rate: 0.0002
  FP16: False


In [18]:
# Initialize trainer
trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=split_dataset['train'],
    eval_dataset=split_dataset['test'],
    formatting_func=format_instruction,
    peft_config=lora_config,
)

print("Starting training...")
print(f"Training on {len(split_dataset['train'])} samples")


Applying formatting function to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Starting training...
Training on 400 samples


In [19]:
# Start training
train_result = trainer.train()

print("\n" + "="*60)
print("Training completed!")
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Training loss: {train_result.metrics['train_loss']:.4f}")
print("="*60)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.205410,1.166217,1.175909,93381.000000,0.702252
100,0.759348,1.229759,0.921814,186762.000000,0.697155



Training completed!
Training time: 1716.17 seconds
Training loss: 1.0586


In [20]:
eval_results = trainer.evaluate()

print("\n" + "="*60)
print("Evaluation on Validation Set Completed!")
print(f"Validation Loss: {eval_results.get('eval_loss', 'N/A'):.4f}")
print("="*60)

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
0.759348,1.166177,100,1.175917,186762.000000,0.702421



Evaluation on Validation Set Completed!
Validation Loss: 1.1662


## 7. Save Fine-tuned Adapter

In [21]:
# Save the LoRA adapter
output_dir = "./medical-chatbot-mistral-lora-final"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✓ LoRA adapter saved to: {output_dir}")
print("\nSaved files:")
import os
for file in os.listdir(output_dir):
    size = os.path.getsize(os.path.join(output_dir, file)) / 1024
    print(f"  - {file}: {size:.2f} KB")


"""
adapter_model.safetensors: weights of finetuning (by lora)

tokenizer.json , tokenizer_config.json: for tokenization



"""

✓ LoRA adapter saved to: ./medical-chatbot-mistral-lora-final

Saved files:
  - README.md: 5.11 KB
  - adapter_config.json: 1.14 KB
  - adapter_model.safetensors: 163898.67 KB
  - chat_template.jinja: 3.87 KB
  - tokenizer.json: 3585.91 KB
  - tokenizer_config.json: 0.45 KB


## 8. Compare Responses: Before vs After Fine-tuning

In [22]:
# Test the fine-tuned model
print("="*80)
print("RESPONSE AFTER FINE-TUNING (Medical-Specialized Mistral)")
print("="*80)

response_after = generate_response(model, tokenizer, test_question)

print(f"\nQuestion: {test_question}\n")
print(f"Response:\n{response_after}")
print("="*80)

RESPONSE AFTER FINE-TUNING (Medical-Specialized Mistral)

Question: If suddenly when I walk my stomach started to hurt me, what do you think it is and what is your suggestion?

Response:
If suddenly when I walk my stomach started to hurt me, what do you think it is and what is your suggestion?  If you experience stomach pain while walking, it could be due to a condition called intermittent claudication. This condition is often associated with peripheral artery disease, where there is a narrowing of the blood vessels that supply blood to the legs. The pain you feel is typically due to a lack of blood flow to the muscles in your legs, which can also affect the stomach due to the close proximity of the abdominal aorta to the legs.

To address this issue, it's important to manage your overall cardiovascular health. This includes maintaining a healthy diet, engaging in regular exercise, and managing stress. If the pain persists or worsens, it's crucial to consult a healthcare professional. 

In [23]:
# Side-by-side comparison
print("\n" + "="*80)
print("COMPARISON: BEFORE vs AFTER FINE-TUNING")
print("="*80)
print(f"\n📝 Question: {test_question}")
print("\n" + "-"*80)
print("🔴 BEFORE FINE-TUNING (General Mistral):")
print("-"*80)
print(response_before)
print("\n" + "-"*80)
print("🟢 AFTER FINE-TUNING (Medical-Specialized Mistral):")
print("-"*80)
print(response_after)
print("="*80)


COMPARISON: BEFORE vs AFTER FINE-TUNING

📝 Question: If suddenly when I walk my stomach started to hurt me, what do you think it is and what is your suggestion?

--------------------------------------------------------------------------------
🔴 BEFORE FINE-TUNING (General Mistral):
--------------------------------------------------------------------------------
If suddenly when I walk my stomach started to hurt me, what do you think it is and what is your suggestion?  I'm not a doctor, but I can suggest some common causes of stomach pain and advise you to seek medical attention if the pain persists. Here are some possibilities:

1. Indigestion or gastritis: This can be caused by eating spicy or fatty foods, stress, or certain medications. Symptoms may include bloating, nausea, and a burning sensation in the stomach.

2. Food poisoning: Consuming contaminated food or water can lead to food poisoning, which can cause stomach pain, diarrhea, vomiting, and fever.

3. Ulcers: Stomach ulcer

In [24]:
# Test with additional medical questions from the dataset domain
additional_questions = [
    "What are the symptoms of diabetes?",
    "How is hypertension treated?",
    "What causes chest pain?"
]

print("\n" + "="*80)
print("TESTING WITH ADDITIONAL MEDICAL QUESTIONS")
print("="*80)

for i, question in enumerate(additional_questions, 1):
    print(f"\n{i}. Question: {question}")
    print("-"*80)
    response = generate_response(model, tokenizer, question, max_new_tokens=200)
    print(f"Response: {response}")
    print("="*80)


TESTING WITH ADDITIONAL MEDICAL QUESTIONS

1. Question: What are the symptoms of diabetes?
--------------------------------------------------------------------------------
Response: What are the symptoms of diabetes?  The symptoms of diabetes include:

1. Frequent urination: This occurs due to the excess sugar in the blood, which draws water from the body, leading to increased urination.

2. Excessive thirst: This is a result of the increased urination, as the body tries to replace the lost fluids.

3. Increased hunger: The body may not be able to effectively use glucose for energy, leading to increased hunger.

4. Fatigue: The body may feel weak and tired due to the lack of energy from glucose.

5. Blurred vision: High blood sugar levels can affect the lens of the eye, leading to blurred vision.

6. Slow healing of wounds: High blood sugar levels can impair the body's ability to heal, leading to slow wound healing.

7. Frequent infections: The immune system may be weakened by high

2

## 9. Integration with LangChain

In [27]:
!pip install langchain langchain-community langchain-huggingface langchain_classic

In [31]:
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# Create a pipeline for the fine-tuned model
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.1,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# Create LangChain LLM
llm = HuggingFacePipeline(pipeline=pipe)

print("✓ LangChain integration ready!")

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'pad_token_id', 'top_p', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✓ LangChain integration ready!


In [32]:
# Define prompt template for medical assistant
medical_prompt = PromptTemplate(
    input_variables=["question"],
    template="""[INST] You are a helpful medical assistant. Provide informative, accurate, and empathetic responses about health concerns. Always remind users to consult healthcare professionals for specific medical advice.

Question: {question} [/INST]"""
)

# Create output parser for clean formatting
output_parser = StrOutputParser()

# Create chain: User -> LLM -> Output Parser
chain = (
    {"question": RunnablePassthrough()}
    | medical_prompt
    | llm
    | output_parser
)

print("✓ LangChain chain created!")
print("\nFlow: User Question → LLM → Output Parser → Formatted Response")

✓ LangChain chain created!

Flow: User Question → LLM → Output Parser → Formatted Response


In [35]:
# Test LangChain integration
def ask_medical_question(question):
    """Ask a medical question and get a formatted response"""
    print(f"\n{'='*80}")
    print(f"🩺 Medical Assistant")
    print(f"{'='*80}")
    print(f"\n❓ Question: {question}\n")
    print(f"💬 Response:")
    print("-"*80)

    response = chain.invoke(question)

    # Clean up response
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()

    print(response)
    print(f"{'='*80}\n")
    return response

In [36]:
# Test with your original question
response = ask_medical_question(test_question)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🩺 Medical Assistant

❓ Question: If suddenly when I walk my stomach started to hurt me, what do you think it is and what is your suggestion?

💬 Response:
--------------------------------------------------------------------------------
If you experience stomach pain while walking, it could be due to a condition called intermittent claudication. This condition is often associated with peripheral artery disease, where blood flow to the legs is restricted. The pain typically occurs when you walk and subsides with rest.

To manage this pain, it's important to maintain a healthy lifestyle, including regular exercise, a balanced diet, and avoiding smoking. If the pain persists or worsens, it's crucial to consult a healthcare professional for further evaluation and treatment.



## 10. Save and Download Model

In [37]:
# Save complete model info
import json

model_info = {
    "base_model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "training_samples": len(split_dataset['train']),
    "lora_rank": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "training_epochs": training_args.num_train_epochs,
    "trainable_parameters": f"{trainable_params:,}",
    "total_parameters": f"{total_params:,}",
    "trainable_percentage": f"{100 * trainable_params / total_params:.4f}%"
}

with open(f"{output_dir}/model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

print("Model information saved!")
print(json.dumps(model_info, indent=2))

Model information saved!
{
  "base_model": "mistralai/Mistral-7B-Instruct-v0.3",
  "dataset": "FreedomIntelligence/medical-o1-reasoning-SFT",
  "training_samples": 400,
  "lora_rank": 16,
  "lora_alpha": 32,
  "training_epochs": 2,
  "trainable_parameters": "41,943,040",
  "total_parameters": "3,800,305,664",
  "trainable_percentage": "1.1037%"
}


In [ ]:
# Download the fine-tuned adapter
!zip -r medical-chatbot-mistral-lora.zip {output_dir}

from google.colab import files
files.download("medical-chatbot-mistral-lora.zip")

print("✓ Model ready for download!")

## 11. How to Load the Fine-tuned Model Later

In [ ]:
# Code to load your fine-tuned model in future sessions

def load_finetuned_mistral(base_model_name, adapter_path):
    """Load base Mistral model and apply fine-tuned LoRA adapter"""

    # Load base model
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    # Load LoRA adapter
    model = PeftModel.from_pretrained(base_model, adapter_path)

    return model, tokenizer

# Example usage:
# model, tokenizer = load_finetuned_mistral(
#     "mistralai/Mistral-7B-Instruct-v0.3",
#     "./medical-chatbot-mistral-lora-final"
# )

## Summary

✅ **What we accomplished:**
1. Loaded Mistral-7B-Instruct with 4-bit quantization
2. Downloaded a real medical Q&A dataset from Hugging Face
3. Prepared and formatted the dataset for training
4. Applied LoRA adapters (efficient fine-tuning)
5. Fine-tuned the model on medical Q&A data
6. Saved the lightweight adapter (~25-30 MB)
7. Compared responses before and after fine-tuning
8. Integrated with LangChain for production-ready usage

**Dataset Used:**
- medalpaca/medical_meadow_medqa (or your chosen dataset)
- Contains real medical Q&A pairs
- Automatically downloaded from Hugging Face

**Why Mistral?**
- No license approval required (fully open source)
- Excellent performance for its size
- Great for conversational tasks

**Alternative Datasets to Try:**
- `medalpaca/medical_meadow_medmcqa` - Multiple choice medical questions
- `GBaker/MedQA` - USMLE-style questions
- `pubmed_qa` - PubMed-based Q&A

**Next Steps:**
- Increase MAX_SAMPLES for more training data
- Experiment with different LoRA ranks
- Try different datasets
- Deploy using FastAPI or Streamlit